# [KHANH] Handling Class Imbalance & Feature Selection

**Mục tiêu:** Xử lý mất cân bằng lớp, chuẩn hóa dữ liệu và chọn 18 đặc trưng quan trọng.

**Đầu vào:** `data/processed/cleaned.csv`  
**Đầu ra:** `data/processed/X_train.npy`, `X_test.npy`, `y_train.npy`, `y_test.npy`, `models/scaler.pkl`, `models/label_encoder.pkl`

## [KHANH] - Bước 1: Import thư viện

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import json

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Thư viện đã được tải thành công.')

Thư viện đã được tải thành công.


## [KHANH] - Bước 2: Tải dữ liệu sạch

In [ ]:
NEEDED_COLS = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Mean',
    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s',
    'Packet Length Mean', 'Packet Length Std', 'SYN Flag Count',
    'ACK Flag Count', 'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'URG Flag Count', 'Label'
]

df = pd.read_csv(
    PROCESSED_DIR / 'cleaned.csv',
    usecols=NEEDED_COLS,
    engine='c',
    on_bad_lines='skip',
    encoding='utf-8',
    encoding_errors='replace'
)
print(f'Shape sau khi tải: {df.shape}')
print(f'Cột: {df.columns.tolist()}')
print(f'Bộ nhớ dùng: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

## [KHANH] - Bước 3: Mã hóa nhãn (Label Encoding)

In [ ]:
le = LabelEncoder()
df['Label_encoded'] = le.fit_transform(df['Label'])

# Lưu ánh xạ nhãn để tra cứu sau
label_mapping = {int(code): name for code, name in enumerate(le.classes_)}
print('Ánh xạ nhãn (mã → tên):')
for code, name in sorted(label_mapping.items()):
    count = (df['Label_encoded'] == code).sum()
    print(f'  {code:2d}: {name:<35} ({count:>8,} mẫu)')

# Lưu mapping ra file JSON để tham khảo
with open(MODELS_DIR / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)
print(f'\nĐã lưu label_mapping.json')

## [KHANH] - Bước 4: Chọn 18 đặc trưng & tách X, y

In [ ]:
SELECTED_FEATURES = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Mean',
    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s',
    'Packet Length Mean', 'Packet Length Std', 'SYN Flag Count',
    'ACK Flag Count', 'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'URG Flag Count'
]

missing_cols = [c for c in SELECTED_FEATURES if c not in df.columns]
if missing_cols:
    print(f'CẢNH BÁO — Cột thiếu: {missing_cols}')
    SELECTED_FEATURES = [c for c in SELECTED_FEATURES if c in df.columns]

X = df[SELECTED_FEATURES].values
y = df['Label_encoded'].values

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Số đặc trưng: {len(SELECTED_FEATURES)}')

## [KHANH] - Bước 5: Chia tập train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, y_test: {y_test.shape}')

# Phân phối nhãn trong tập train
unique, counts = np.unique(y_train, return_counts=True)
print('\nPhân phối nhãn trong tập TRAIN:')
for u, c in zip(unique, counts):
    print(f'  {label_mapping[u]:<35}: {c:>8,}')

## [KHANH] - Bước 6: Chuẩn hóa dữ liệu (StandardScaler)

In [ ]:
# Fit chỉ trên tập train, transform cả train lẫn test
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print('Đã chuẩn hóa dữ liệu (StandardScaler).')
print(f'X_train mean ≈ {X_train.mean():.4f}, std ≈ {X_train.std():.4f}')
print(f'X_test  mean ≈ {X_test.mean():.4f},  std ≈ {X_test.std():.4f}')

## [KHANH] - Bước 7: Xử lý mất cân bằng lớp (SMOTE + RandomUnderSampler)

In [ ]:
from collections import Counter

print('Phân phối nhãn TRƯỚC khi cân bằng:')
counter_before = Counter(y_train)
for label_idx in sorted(counter_before):
    print(f'  {label_mapping[label_idx]:<35}: {counter_before[label_idx]:>8,}')

# --- Tính sampling strategy cho SMOTE ---
# Đưa các lớp thiểu số lên 10% số lượng lớp đa số
majority_count = max(counter_before.values())
target_minority = max(int(majority_count * 0.10), 10)  # tối thiểu 10 mẫu

smote_strategy = {}
for label_idx, count in counter_before.items():
    if count < target_minority:
        smote_strategy[label_idx] = target_minority

print(f'\nLớp đa số có {majority_count:,} mẫu — mục tiêu SMOTE: {target_minority:,}')
print(f'Số lớp cần oversample: {len(smote_strategy)}')

# Áp dụng SMOTE nếu có lớp cần oversample
if smote_strategy:
    smote = SMOTE(sampling_strategy=smote_strategy, random_state=42, k_neighbors=5)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print(f'\nSau SMOTE — X_train: {X_train.shape}')
else:
    print('\nKhông cần SMOTE — các lớp đã đủ mẫu.')

# --- RandomUnderSampler: giảm BENIGN để cân bằng ---
counter_after_smote = Counter(y_train)
current_majority = max(counter_after_smote.values())
target_majority = int(current_majority * 0.5)  # giảm 50% BENIGN

under_strategy = {}
for label_idx, count in counter_after_smote.items():
    if count == current_majority and count > target_majority:
        under_strategy[label_idx] = target_majority

if under_strategy:
    rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
    X_train, y_train = rus.fit_resample(X_train, y_train)
    print(f'Sau RandomUnderSampler — X_train: {X_train.shape}')

print('\nPhân phối nhãn SAU khi cân bằng:')
counter_after = Counter(y_train)
for label_idx in sorted(counter_after):
    print(f'  {label_mapping[label_idx]:<35}: {counter_after[label_idx]:>8,}')

## [KHANH] - Bước 8: Lưu dữ liệu và artifacts

In [ ]:
# Lưu arrays
np.save(PROCESSED_DIR / 'X_train.npy', X_train)
np.save(PROCESSED_DIR / 'X_test.npy',  X_test)
np.save(PROCESSED_DIR / 'y_train.npy', y_train)
np.save(PROCESSED_DIR / 'y_test.npy',  y_test)

# Lưu scaler và label_encoder
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
joblib.dump(le,     MODELS_DIR / 'label_encoder.pkl')

print('Đã lưu:')
print(f'  data/processed/X_train.npy   {X_train.shape}')
print(f'  data/processed/X_test.npy    {X_test.shape}')
print(f'  data/processed/y_train.npy   {y_train.shape}')
print(f'  data/processed/y_test.npy    {y_test.shape}')
print(f'  models/scaler.pkl')
print(f'  models/label_encoder.pkl')
print('\n✓ [KHANH] Hoàn thành Preprocessing Pipeline!')